# Peakflo Expense Classification Report

This notebook is the end-to-end report for the Peakflo take-home task. It documents the business problem, definitions, dataset structure, feature engineering, model training, split strategy, overfitting diagnosis, evaluation results, and deployment implications.

## Objectives

1. Predict the correct `accountName` for each expense line item.
2. Reach at least 85% accuracy with a method that is practical, explainable, and reproducible.
3. Evaluate the model with a split strategy that reflects real unseen transaction patterns.

## Deliverable Structure

- Definitions and business framing
- Dataset loading and exploratory data analysis
- Feature engineering and model design
- Train/test split strategy and its justification
- Hyperparameter tuning and overfitting diagnosis
- Final metrics, error analysis, and business conclusions


## Business Problem Definition

Expense classification is a **multiclass text classification** problem:

- **Input**: vendor ID, item name, item description, transaction amount
- **Target**: accounting account name
- **Goal**: assign the correct account as accurately as possible

### Definitions

- **Class imbalance**: some labels have far more examples than others.
- **Overfitting**: the model performs much better on training data than on unseen data.
- **Underfitting**: the model performs poorly on both training and unseen data.
- **Leakage**: information from train data accidentally appears in test data and inflates metrics.
- **Macro F1**: average F1 over classes, giving rare classes equal weight.
- **Weighted F1**: average F1 weighted by class frequency.

### Why This Matters

A model with high overall accuracy but poor rare-class behavior can still create operational problems in finance workflows. That is why this report shows both natural-distribution and balanced unseen benchmarks.


## End-to-End Pipeline Diagram

```mermaid
flowchart TD
    A[accounts-bills.json] --> B[Load with pandas]
    B --> C[Normalize text and amounts]
    C --> D[EDA and data quality checks]
    D --> E[Feature engineering]
    E --> F[Word TF-IDF]
    E --> G[Character TF-IDF]
    E --> H[Vendor one-hot]
    E --> I[Log amount scaling]
    F --> J[LinearSVC classifier]
    G --> J
    H --> J
    I --> J
    J --> K[Grouped CV tuning]
    K --> L[Grouped holdout evaluation]
    K --> M[Balanced unseen benchmark]
    L --> N[Error analysis]
    M --> N
    N --> O[Final report and saved model]
```

## ASCII Architecture

```text
JSON dataset
   |
   v
+---------------------------+
| pandas loading / cleanup  |
+---------------------------+
   |
   v
+---------------------------+
| feature engineering       |
| - word tf-idf             |
| - char tf-idf             |
| - vendor one-hot          |
| - log(amount)             |
+---------------------------+
   |
   v
+---------------------------+
| LinearSVC                 |
+---------------------------+
   |
   v
+---------------------------+
| grouped CV tuning         |
+---------------------------+
   |
   +--> grouped natural holdout
   |
   +--> balanced unseen holdout
   |
   v
report + artifacts
```


In [ ]:
import os
os.chdir(r"C:\\Temp\\Interviews\\Peakflo")

In [1]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.feature_engineering_pipeline import load_records
from src.training_pipeline import (
    baseline_lookup_accuracy,
    compute_overfitting_diagnostics,
    evaluate_balanced_holdout,
    evaluate_holdout,
    evaluate_repeated_balanced_holdout,
    evaluate_repeated_splits,
    tune_hyperparameters_grouped_cv,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

DATA_PATH = Path("accounts-bills.json")
SUMMARY_PATH = Path("artifacts/evaluation_summary.json")

records = load_records(DATA_PATH)
df = pd.DataFrame.from_records(record.__dict__ for record in records)
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

print(f"rows={len(df)} labels={df['account_name'].nunique()} vendors={df['vendor_id'].nunique()}")
df.head()


ModuleNotFoundError: No module named 'src'

## Dataset Snapshot

The dataset is loaded through the project pipeline, which already standardizes:

- missing text values
- numeric amount coercion
- normalized item names
- joined text fields used by the model

The working dataframe below is built from the normalized record objects so that the notebook and production code stay aligned.


In [ ]:
dataset_overview = pd.Series({
    "record_count": len(df),
    "unique_vendors": df["vendor_id"].nunique(),
    "unique_account_names": df["account_name"].nunique(),
    "labels_lt_10": int((df["account_name"].value_counts() < 10).sum()),
    "labels_lt_5": int((df["account_name"].value_counts() < 5).sum()),
    "singleton_labels": int((df["account_name"].value_counts() == 1).sum()),
    "missing_descriptions": int(df["item_description"].str.strip().eq("").sum()),
    "min_amount": float(df["item_total_amount"].min()),
    "max_amount": float(df["item_total_amount"].max()),
})
dataset_overview.to_frame("value")


## Exploratory Data Analysis

### Justification

Before modeling, we need to answer:

1. How imbalanced are the classes?
2. Are there obvious quality issues?
3. Are a few vendors dominating the sample?
4. How large is the numerical amount range?

These questions inform model choice and validation strategy.


In [ ]:
label_counts = df["account_name"].value_counts()
top20_labels = label_counts.head(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20_labels.index, top20_labels.values, color="#2f6db2")
ax.set_title("Top 20 Account Labels by Frequency")
ax.set_xlabel("Record Count")
ax.set_ylabel("Account Name")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["item_total_amount"], bins=40, color="#467f52", edgecolor="white")
axes[0].set_title("Raw Amount Distribution")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Count")

axes[1].hist(df["amount_log"], bins=40, color="#9b4d16", edgecolor="white")
axes[1].set_title("Log(1 + |Amount|) Distribution")
axes[1].set_xlabel("log(1 + |amount|)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
vendor_counts = df["vendor_id"].value_counts()
top15_vendors = vendor_counts.head(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15_vendors.index, top15_vendors.values, color="#8d5aa7")
ax.set_title("Top 15 Vendors by Transaction Count")
ax.set_xlabel("Record Count")
ax.set_ylabel("Vendor ID")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bucket_labels = ["< 3", "< 5", "< 10", ">= 10"]
bucket_values = [
    int((label_counts < 3).sum()),
    int(((label_counts >= 3) & (label_counts < 5)).sum()),
    int(((label_counts >= 5) & (label_counts < 10)).sum()),
    int((label_counts >= 10).sum()),
]
ax.bar(bucket_labels, bucket_values, color=["#b23a48", "#d97d54", "#e9b44c", "#2a9d8f"])
ax.set_title("Class Frequency Buckets")
ax.set_xlabel("Bucket")
ax.set_ylabel("Number of Labels")
plt.tight_layout()
plt.show()


### EDA Interpretation

- The problem is **heavily imbalanced**.
- The amount scale is wide, so log scaling is justified.
- Text fields carry the dominant signal, but vendor identity also contributes.
- Many labels have very little support, which makes macro metrics important.


## Feature Engineering

### Design Choice

The final model uses a sparse linear text stack:

- **Word TF-IDF (1-2 grams)**: captures token-level semantics and short phrases.
- **Character TF-IDF (3-5 grams)**: captures spelling variants, abbreviations, and formatting patterns.
- **Vendor one-hot encoding**: captures supplier-specific account behavior.
- **Log-scaled amount**: injects a simple numeric feature without dominating sparse text features.

### Justification

This is a practical choice for a medium-sized, text-heavy tabular dataset:

- fast to train
- interpretable enough for debugging
- strong baseline for multiclass sparse data
- less operationally risky than a heavier neural approach


In [ ]:
feature_engineering_snippet = """
FeatureUnion(
    [
        ("word_tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        ("vendor", OneHotEncoder(handle_unknown="ignore")),
        ("amount", MaxAbsScaler()),
    ]
)
"""
print(feature_engineering_snippet)


## Train / Test Split Strategy

### Why Random Row Splits Are Not Enough

If identical or near-identical `itemName` patterns appear in both train and test, row-level random splits produce optimistic metrics. In this dataset, repeated transaction templates are common.

### Primary Validation Strategy

The primary score uses **grouped splits by normalized item name**:

- all rows for the same normalized item name stay together
- this reduces text leakage
- it better simulates unseen transaction templates

### Balanced Benchmark

A second evaluation constructs a **balanced unseen benchmark**:

- equal samples per eligible class
- held-out item-name groups
- used to stress-test rare-class behavior

This is not a replacement for production-like natural-distribution evaluation. It is a complementary fairness/stress metric.


In [ ]:
split_strategy_table = pd.DataFrame(
    [
        ["Random holdout", "Optimistic", "Quick baseline only"],
        ["Grouped holdout", "Best production-like estimate", "Primary headline metric"],
        ["Balanced unseen holdout", "Class fairness stress test", "Secondary benchmark"],
    ],
    columns=["Strategy", "Bias Risk", "Use Case"],
)
split_strategy_table


## Baseline And Tuned Model Evaluation


In [ ]:
baseline_acc = baseline_lookup_accuracy(records)

tuning = tune_hyperparameters_grouped_cv(
    records,
    C_values=[0.5, 1.0, 2.0, 4.0, 8.0],
    class_weights=[None, "balanced"],
    oversample_min_count_values=[0, 5, 10],
)

default_group = evaluate_holdout(records, strategy="group_item")
tuned_group = evaluate_holdout(
    records,
    strategy="group_item",
    C=tuning["best_C"],
    class_weight=tuning["best_class_weight"],
    oversample_min_count=tuning["best_oversample_min_count"],
)

balanced_tuned = evaluate_balanced_holdout(
    records,
    C=tuning["best_C"],
    class_weight=tuning["best_class_weight"],
    oversample_min_count=tuning["best_oversample_min_count"],
)

pd.DataFrame(
    [
        ["Exact item/vendor lookup baseline", baseline_acc, None, None],
        ["Grouped holdout default", default_group["accuracy"], default_group["macro_f1"], default_group["accuracy_gap"]],
        ["Grouped holdout tuned", tuned_group["accuracy"], tuned_group["macro_f1"], tuned_group["accuracy_gap"]],
        ["Balanced unseen tuned", balanced_tuned["accuracy"], balanced_tuned["macro_f1"], balanced_tuned["accuracy_gap"]],
    ],
    columns=["Model / Benchmark", "Accuracy", "Macro F1", "Train-Test Gap"],
)


## Hyperparameter Tuning Grid

The tuning search is grouped by normalized item name to avoid leakage during model selection.

### What Was Tuned

- `C`: margin regularization strength for LinearSVC
- `class_weight`
- `oversample_min_count`

### Why This Matters

Selecting hyperparameters on a random split would let leakage influence model selection. Grouped CV makes the tuning process consistent with the deployment objective.


In [ ]:
tuning_df = pd.DataFrame(summary["hyperparameter_tuning"]["grid"])
tuning_df = tuning_df.sort_values(["oversample_min_count", "class_weight", "C"]).reset_index(drop=True)
tuning_df.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, group in tuning_df.groupby(["class_weight", "oversample_min_count"]):
    group = group.sort_values("C")
    ax.plot(
        group["C"],
        group["val_accuracy_mean"],
        marker="o",
        linewidth=2,
        label=f"class_weight={label[0]}, oversample={label[1]}",
    )

ax.set_xscale("log", base=2)
ax.set_title("Grouped CV Validation Accuracy by Hyperparameter Setting")
ax.set_xlabel("C (log2 scale)")
ax.set_ylabel("Mean Validation Accuracy")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, group in tuning_df.groupby(["class_weight", "oversample_min_count"]):
    group = group.sort_values("C")
    ax.plot(
        group["C"],
        group["gap_mean"],
        marker="o",
        linewidth=2,
        label=f"class_weight={label[0]}, oversample={label[1]}",
    )

ax.set_xscale("log", base=2)
ax.set_title("Grouped CV Train-Validation Gap by Hyperparameter Setting")
ax.set_xlabel("C (log2 scale)")
ax.set_ylabel("Mean Gap")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## Overfitting And Learning-Curve Diagnosis

### Interpretation Logic

- If training accuracy is high and validation is much lower, the model is overfitting.
- If both are low, the model is underfitting.
- If validation improves with more data while training stays high, more data would likely help.


In [ ]:
diagnostics = summary["diagnostics"]
learning_df = pd.DataFrame(diagnostics["learning_curve"])
validation_df = pd.DataFrame(diagnostics["validation_curve"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(learning_df["train_size"], learning_df["train_accuracy"], marker="o", label="Train")
axes[0].plot(learning_df["train_size"], learning_df["validation_accuracy"], marker="o", label="Validation")
axes[0].set_title("Learning Curve")
axes[0].set_xlabel("Training Size")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(validation_df["C"], validation_df["train_accuracy"], marker="o", label="Train")
axes[1].plot(validation_df["C"], validation_df["validation_accuracy"], marker="o", label="Validation")
axes[1].set_xscale("log", base=2)
axes[1].set_title("Validation Curve")
axes[1].set_xlabel("C")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame([diagnostics["fit_assessment"]]).T.rename(columns={0: "value"})


### Diagnostic Conclusion

The model is **not underfitting**. The validation curve and learning curve both show that:

- training accuracy remains very high
- validation accuracy is also high, but lower than training
- the gap is moderate, which confirms overfitting
- more data would likely help more than further regularization alone


## Error Analysis By Class Frequency

This section asks a critical question:

> Is the model failing mostly on rare classes or on frequent classes?

If the largest errors are concentrated in rare classes, the issue is more about data sparsity than model architecture.


In [ ]:
error_default = pd.DataFrame(summary["holdout"]["group_item"]["error_analysis_by_frequency"]).T
error_tuned = pd.DataFrame(summary["holdout"]["group_item_tuned"]["error_analysis_by_frequency"]).T

display(error_default)
display(error_tuned)


In [ ]:
freq_order = [
    "singleton (n=1)",
    "very_rare (2-4)",
    "rare (5-19)",
    "medium (20-99)",
    "frequent (100+)",
]

default_acc = [summary["holdout"]["group_item"]["error_analysis_by_frequency"][k]["accuracy"] for k in freq_order]
tuned_acc = [summary["holdout"]["group_item_tuned"]["error_analysis_by_frequency"][k]["accuracy"] for k in freq_order]

x = np.arange(len(freq_order))
width = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, default_acc, width, label="Default", color="#6c8ebf")
ax.bar(x + width/2, tuned_acc, width, label="Tuned", color="#93c47d")
ax.set_xticks(x)
ax.set_xticklabels(freq_order, rotation=20)
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy by Class-Frequency Bucket")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()


## Final Metrics Summary


In [ ]:
final_metrics = pd.DataFrame(
    [
        [
            "Default grouped holdout",
            summary["holdout"]["group_item"]["accuracy"],
            summary["holdout"]["group_item"]["macro_f1"],
            summary["holdout"]["group_item"]["accuracy_gap"],
        ],
        [
            "Tuned grouped holdout",
            summary["holdout"]["group_item_tuned"]["accuracy"],
            summary["holdout"]["group_item_tuned"]["macro_f1"],
            summary["holdout"]["group_item_tuned"]["accuracy_gap"],
        ],
        [
            "Tuned repeated grouped mean",
            summary["repeated"]["group_item_tuned"]["accuracy_mean"],
            summary["repeated"]["group_item_tuned"]["macro_f1_mean"],
            summary["repeated"]["group_item_tuned"]["accuracy_gap_mean"],
        ],
        [
            "Tuned balanced repeated mean",
            summary["balanced_repeated"]["tuned_model"]["accuracy_mean"],
            summary["balanced_repeated"]["tuned_model"]["macro_f1_mean"],
            summary["balanced_repeated"]["tuned_model"]["accuracy_gap_mean"],
        ],
    ],
    columns=["Metric", "Accuracy", "Macro F1", "Gap"],
)
final_metrics


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    final_metrics["Metric"],
    final_metrics["Accuracy"],
    color=["#9e2a2b", "#386641", "#4ea8de", "#f4a261"],
)
ax.set_ylim(0.75, 0.95)
ax.set_title("Comparison of Final Accuracy Metrics")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## Business Recommendation

### What Should Be Reported

1. **Primary model score**: tuned grouped holdout accuracy
2. **Robustness score**: repeated grouped mean and standard deviation
3. **Fairness stress test**: balanced unseen benchmark
4. **Risk note**: rare classes remain the main source of residual error

### Production Implication

This model is deployable as a human-assist classifier:

- high overall accuracy
- explainable sparse features
- strong grouped unseen performance
- known limitation on rare labels

### Recommended Next Improvements

1. collect more examples for low-support labels
2. create label-taxonomy features or account hierarchies
3. add confidence-based review routing for uncertain predictions
4. consider vendor-history features if production policy allows them


## Reproducibility

This notebook is aligned with the repository codebase:

- data loading: `src/feature_engineering_pipeline.py`
- training and evaluation: `src/training_pipeline.py`
- saved metrics: `artifacts/evaluation_summary.json`

The code cells above are intentionally explicit so the report can be read linearly and understood without jumping back and forth between files.


# Appendix A: Mathematical Notes

This section records the main mathematical ideas used by the pipeline.

## A.1 TF-IDF

For term $t$ in document $d$:

$$
\mathrm{tfidf}(t, d) = \mathrm{tf}(t, d) \times \mathrm{idf}(t)
$$

where inverse document frequency is:

$$
\mathrm{idf}(t) = \log\left(\frac{N + 1}{\mathrm{df}(t) + 1}\right) + 1
$$

- $N$ is the number of documents.
- $\mathrm{df}(t)$ is the number of documents containing term $t$.

### Why This Is Used

TF-IDF is appropriate here because expense descriptions contain short but highly informative phrases. TF-IDF emphasizes terms that are distinctive for a transaction description while down-weighting text that appears everywhere.

## A.2 Linear SVM Scoring Rule

For class $k$, the linear classifier computes:

$$
f_k(x) = w_k^T x + b_k
$$

The predicted label is:

$$
\hat{y} = \arg\max_k f_k(x)
$$

### Optimization View

The linear SVM objective can be written as:

$$
\min_{w,b} \; \frac{1}{2}\lVert w \rVert^2 + C \sum_{i=1}^{n} \max\left(0, 1 - y_i(w^T x_i + b)\right)
$$

where:

- $\lVert w \rVert^2$ penalizes model complexity
- $C$ controls the regularization strength

### Interpretation

- lower $C$ means stronger regularization
- higher $C$ means weaker regularization and more risk of memorization

## A.3 Precision, Recall, and F1

For a class $c$:

$$
\mathrm{Precision}_c = \frac{TP_c}{TP_c + FP_c}
$$

$$
\mathrm{Recall}_c = \frac{TP_c}{TP_c + FN_c}
$$

$$
F1_c = \frac{2 \cdot \mathrm{Precision}_c \cdot \mathrm{Recall}_c}{\mathrm{Precision}_c + \mathrm{Recall}_c}
$$

Macro F1 is:

$$
\mathrm{MacroF1} = \frac{1}{K} \sum_{c=1}^{K} F1_c
$$

Weighted F1 is:

$$
\mathrm{WeightedF1} = \sum_{c=1}^{K} \frac{n_c}{n} F1_c
$$

### Why Both Are Reported

- Macro F1 is needed because rare classes matter.
- Weighted F1 is needed because the production distribution is imbalanced.

## A.4 Generalization Gap

The train-validation gap is:

$$
\mathrm{gap} = \mathrm{train\ accuracy} - \mathrm{validation\ accuracy}
$$

For the tuned grouped holdout:

$$
0.9924 - 0.9013 \approx 0.0911
$$

That is the main numerical evidence of moderate overfitting.


# Appendix A.5 Result And Accuracy Metric Definitions

This subsection defines the result metrics reported throughout the notebook and in the `classification_report` output.

## A.5.1 Holdout Accuracy

Holdout accuracy is the fraction of correct predictions on an unseen evaluation set:

$$
\mathrm{Accuracy}=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(\hat{y}_i=y_i)
$$

where:

- $n$ is the number of evaluated samples
- $\hat{y}_i$ is the predicted class for sample $i$
- $y_i$ is the true class for sample $i$
- $\mathbf{1}(\cdot)$ equals 1 when the condition is true and 0 otherwise

### Interpretation

- grouped holdout accuracy is the main production-like metric in this project
- random holdout accuracy is optimistic because it allows phrase-pattern leakage
- balanced holdout accuracy is a fairness stress test, not the production headline score

## A.5.2 Precision

Precision for class $c$ measures how many predictions of class $c$ were actually correct:

$$
\mathrm{Precision}_c=\frac{TP_c}{TP_c+FP_c}
$$

where $TP_c$ is true positives and $FP_c$ is false positives for class $c$.

### Interpretation

- high precision means the model is conservative and usually right when it predicts a class
- low precision means the model is over-predicting that class and producing many false alarms

## A.5.3 Recall

Recall for class $c$ measures how many actual rows of class $c$ the model successfully recovered:

$$
\mathrm{Recall}_c=\frac{TP_c}{TP_c+FN_c}
$$

where $FN_c$ is false negatives for class $c$.

### Interpretation

- high recall means most true rows of the class are found
- low recall means the model is missing many rows that belong to that class

## A.5.4 F1 Score

F1 for class $c$ is the harmonic mean of precision and recall:

$$
F1_c = 2\cdot\frac{\mathrm{Precision}_c\cdot\mathrm{Recall}_c}{\mathrm{Precision}_c+\mathrm{Recall}_c}
$$

### Interpretation

- F1 is high only when both precision and recall are high
- it is useful when false positives and false negatives both matter

## A.5.5 Support

Support is the number of true examples belonging to class $c$ in the evaluation data:

$$
\mathrm{Support}_c = n_c
$$

### Interpretation

- support explains how much data backs each class-level metric
- low-support classes produce noisier precision, recall, and F1 estimates

## A.5.6 Macro F1

Macro F1 averages class-level F1 scores equally:

$$
\mathrm{MacroF1}=\frac{1}{K}\sum_{c=1}^{K} F1_c
$$

where $K$ is the number of classes.

### Interpretation

Macro F1 matters because rare labels should not disappear inside a strong overall accuracy score.

## A.5.7 Weighted F1

Weighted F1 averages class-level F1 scores using class support:

$$
\mathrm{WeightedF1}=\sum_{c=1}^{K}\frac{n_c}{n}F1_c
$$

where $n_c$ is the number of samples in class $c$ and $n=\sum_{c=1}^{K} n_c$.

### Interpretation

Weighted F1 is more aligned with the natural production distribution than Macro F1.

## A.5.8 Macro Precision And Macro Recall

The macro averages reported by `classification_report` are:

$$
\mathrm{MacroPrecision}=\frac{1}{K}\sum_{c=1}^{K}\mathrm{Precision}_c
$$

$$
\mathrm{MacroRecall}=\frac{1}{K}\sum_{c=1}^{K}\mathrm{Recall}_c
$$

These give every class equal weight, regardless of how frequent that class is.

## A.5.9 Weighted Precision And Weighted Recall

The weighted averages reported by `classification_report` are:

$$
\mathrm{WeightedPrecision}=\sum_{c=1}^{K}\frac{n_c}{n}\mathrm{Precision}_c
$$

$$
\mathrm{WeightedRecall}=\sum_{c=1}^{K}\frac{n_c}{n}\mathrm{Recall}_c
$$

These weight each class by its observed frequency in the evaluation set.

## A.5.10 Repeated Mean Accuracy

When evaluation is repeated across multiple splits, the reported mean accuracy is:

$$
\bar{A}=\frac{1}{m}\sum_{j=1}^{m} A_j
$$

where $A_j$ is the accuracy from split $j$ and $m$ is the number of repeated splits.

### Why It Is Used

A single holdout split can be noisy. Repeated grouped evaluation gives a more stable estimate of expected performance.

## A.5.11 Accuracy Standard Deviation

The standard deviation of repeated accuracies is:

$$
\sigma_A = \sqrt{\frac{1}{m}\sum_{j=1}^{m}(A_j-\bar{A})^2}
$$

### Interpretation

- lower standard deviation means more stable evaluation across splits
- higher standard deviation means the score depends more on the particular split

## A.5.12 Train-Test Gap

The train-test gap is:

$$
\mathrm{Gap}=\mathrm{TrainAccuracy}-\mathrm{ValidationAccuracy}
$$

### Interpretation

- small gap suggests better generalization
- large gap suggests overfitting
- in this project, the tuned grouped gap is around $0.0911$

## A.5.13 Balanced Unseen Accuracy

Balanced unseen accuracy uses the same accuracy formula as holdout accuracy, but on a deliberately class-balanced evaluation set:

$$
\mathrm{BalancedAccuracyLikeHoldout}=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(\hat{y}_i=y_i)
$$

The difference is not in the formula, but in the construction of the evaluation sample: each eligible class contributes the same number of unseen rows.

### Interpretation

This is useful for judging rare-class robustness, but it should not replace the grouped natural-distribution score as the main business metric.

## A.5.14 Result Table Reading Guide

A professional reading of the result table in this project is:

- use grouped holdout accuracy as the primary business-facing estimate
- use repeated grouped mean and standard deviation to judge stability
- use macro metrics to judge rare-class behavior
- use weighted metrics to judge natural-distribution behavior
- use the train-validation gap to judge overfitting
- use balanced unseen accuracy as a secondary stress test for class fairness


# Appendix B: Indexed Library Reference

The definitions below are paraphrased from official project documentation and are included here to justify each library choice.

## B.1 pandas

- **Documented definition**: pandas provides fast, flexible, and expressive data structures for working with labeled or relational data.
- **Used in this project for**: JSON loading, missing-value cleanup, aggregation, and summary tables.
- **Why justified**: this assignment is structured tabular text data, so dataframe operations are the cleanest and most readable way to perform EDA and reporting.

## B.2 NumPy

- **Documented definition**: NumPy is the fundamental package for scientific computing in Python and provides multidimensional arrays plus numerical routines.
- **Used in this project for**: vectorized labels, indexing, numerical summaries, and compatibility with scikit-learn.
- **Why justified**: scikit-learn expects array-oriented inputs and outputs, so NumPy is the natural numerical base layer.

## B.3 Matplotlib

- **Documented definition**: Matplotlib is a comprehensive library for creating static, animated, and interactive visualizations in Python.
- **Used in this project for**: histograms, bar charts, learning curves, validation curves, and summary plots.
- **Why justified**: a report notebook needs reproducible professional charts, and Matplotlib is the standard plotting tool for that.

## B.4 scikit-learn

- **Documented definition**: scikit-learn provides simple and efficient tools for predictive data analysis.
- **Used in this project for**:
  - TF-IDF vectorization
  - one-hot encoding
  - scaling
  - grouped train/test splitting
  - LinearSVC classification
  - evaluation metrics and tuning
- **Why justified**: this problem is a sparse multiclass text-classification problem, which is a strong fit for classical scikit-learn pipelines.

## B.5 joblib

- **Documented definition**: joblib provides lightweight pipelining tools in Python and efficient persistence for large Python objects, especially those containing NumPy arrays.
- **Used in this project for**: saving and loading the trained model artifact.
- **Why justified**: model persistence is part of reproducibility, and joblib is a practical standard for classical ML pipelines.

## B.6 scipy.sparse.csr_matrix

- **Documented definition**: CSR is a compressed sparse row matrix representation.
- **Used in this project for**: storing high-dimensional sparse text features and scaled numeric features efficiently.
- **Why justified**: TF-IDF matrices are mostly zeros, so dense storage is wasteful in both memory and compute.
- **Mathematical intuition**: if a feature matrix $X \in \mathbb{R}^{n \times p}$ has mostly zero entries, sparse storage keeps only non-zero values and index locations rather than all $n \times p$ cells.

## B.7 sklearn.base.BaseEstimator

- **Documented definition**: base class for estimators in scikit-learn.
- **Used in this project for**: making custom transformers compatible with the scikit-learn estimator API.
- **Why justified**: a consistent estimator interface is required for pipeline composition, fitting, parameter inspection, and validation.

## B.8 sklearn.base.TransformerMixin

- **Documented definition**: mixin class for transformers in scikit-learn that provides a default \`fit_transform\` implementation.
- **Used in this project for**: custom feature transformers that convert record dictionaries into model-ready sparse features.
- **Why justified**: this keeps custom preprocessing aligned with native scikit-learn transformers.

## B.9 sklearn.feature_extraction.text.TfidfVectorizer

- **Documented definition**: convert a collection of raw documents to a matrix of TF-IDF features.
- **Used in this project for**: word-level and character-level sparse text features.
- **Why justified**: item names and descriptions are the dominant predictive signal, and TF-IDF is a strong classical representation for sparse text classification.
- **Mathematics**:

$$
\mathrm{tfidf}(t,d)=\mathrm{tf}(t,d)\cdot\mathrm{idf}(t)
$$

with

$$
\mathrm{idf}(t)=\log\left(\frac{N+1}{\mathrm{df}(t)+1}\right)+1
$$

## B.10 sklearn.pipeline.FeatureUnion

- **Documented definition**: concatenate results of multiple transformer objects.
- **Used in this project for**: combining word TF-IDF, character TF-IDF, vendor encoding, and amount scaling into one feature matrix.
- **Why justified**: the model needs text, categorical, and numeric signals together in a single aligned representation.
- **Mathematical intuition**: if feature blocks are $X_1, X_2, \dots, X_m$, FeatureUnion forms a horizontal concatenation

$$
X = [X_1\; X_2\; \dots \; X_m]
$$

## B.11 sklearn.preprocessing.MaxAbsScaler

- **Documented definition**: scale each feature by its maximum absolute value.
- **Used in this project for**: scaling the log-transformed amount feature without destroying sparsity.
- **Why justified**: the amount feature should be numerically comparable without forcing dense centering.
- **Mathematics**:

$$
x'_j = \frac{x_j}{\max_i |x_{ij}|}
$$

## B.12 sklearn.preprocessing.OneHotEncoder

- **Documented definition**: encode categorical features as a one-hot numeric array.
- **Used in this project for**: converting vendor IDs into binary indicator columns.
- **Why justified**: vendor identity is categorical, not ordinal, so one-hot encoding avoids imposing a false numerical relationship.
- **Mathematical intuition**: for category set $\{c_1,\dots,c_K\}$, one-hot encoding maps category $c_k$ to vector $e_k \in \{0,1\}^K$ with one 1 and the rest 0.

## B.13 sklearn.pipeline.Pipeline

- **Documented definition**: assemble several steps that can be cross-validated together while setting different parameters.
- **Used in this project for**: connecting feature construction and LinearSVC into one trainable estimator.
- **Why justified**: the training, evaluation, and persistence logic should operate on one consistent object.

## B.14 sklearn.metrics.accuracy_score

- **Documented definition**: compute the classification accuracy score.
- **Used in this project for**: headline performance measurement on holdout and repeated splits.
- **Why justified**: the assignment sets an explicit accuracy threshold.
- **Mathematics**:

$$
\mathrm{Accuracy}=\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}(\hat{y}_i=y_i)
$$

## B.15 sklearn.metrics.classification_report

- **Documented definition**: build a text summary showing main classification metrics.
- **Used in this project for**: per-class precision, recall, F1, macro averages, and weighted averages.
- **Why justified**: a multiclass imbalanced problem needs more than a single scalar metric.

## B.16 sklearn.model_selection.GroupShuffleSplit

- **Documented definition**: randomized train/test indices according to a third-party provided group.
- **Used in this project for**: grouped holdout and grouped repeated evaluation by normalized item name.
- **Why justified**: prevents repeated item-name templates from leaking across train and test.

## B.17 sklearn.model_selection.ShuffleSplit

- **Documented definition**: randomized permutation cross-validator.
- **Used in this project for**: quick random-row baseline comparison.
- **Why justified**: useful as a naive benchmark, even though it is optimistic here.

## B.18 sklearn.model_selection.StratifiedGroupKFold

- **Documented definition**: variation of K-fold that preserves class proportions as much as possible while respecting non-overlapping groups.
- **Used in this project for**: feasibility checks on grouped stratified evaluation.
- **Why justified**: helps test whether group-aware evaluation can also remain label-aware under severe imbalance.

## B.19 sklearn.model_selection.learning_curve

- **Documented definition**: determine cross-validated training and test scores for different training set sizes.
- **Used in this project for**: diagnosing whether more data would likely improve grouped validation accuracy.
- **Why justified**: the train-validation gap alone does not show whether data scarcity is the main bottleneck.

## B.20 sklearn.model_selection.validation_curve

- **Documented definition**: determine training and test scores for varying parameter values.
- **Used in this project for**: studying how the SVM regularization parameter $C$ changes train and validation accuracy.
- **Why justified**: necessary for bias-variance diagnosis and hyperparameter interpretation.

## B.21 sklearn.svm.LinearSVC

- **Documented definition**: linear support vector classification implemented with liblinear.
- **Used in this project for**: the final multiclass sparse-text classifier.
- **Why justified**: linear SVMs are strong baselines for high-dimensional sparse text problems and are efficient enough for repeated grouped validation.
- **Mathematics**: for class score $f_k(x)=w_k^T x+b_k$, prediction is

$$
\hat{y}=\arg\max_k f_k(x)
$$

and the optimization balances margin size against hinge-loss penalties through $C$.


# Appendix C: Indexed Concept Reference

The concept definitions below are aligned with standard machine-learning documentation and glossary sources.

## C.1 Feature

- **Definition**: an input variable used by a model.
- **In this project**: text features, vendor identity, and log-scaled amount.

## C.2 Label

- **Definition**: the answer or target portion of a supervised-learning example.
- **In this project**: the label is `accountName`.

## C.3 Labeled Example

- **Definition**: an example containing one or more features and a label.
- **In this project**: one expense record with text, vendor, amount, and the true account label.

## C.4 Validation Set

- **Definition**: the subset used for initial model evaluation during development.
- **In this project**: grouped holdout and grouped CV act as validation mechanisms.

## C.5 Overfitting

- **Definition**: a model performs well on training data but poorly on new unseen data.
- **Observed here**: training accuracy is materially higher than grouped unseen accuracy.

## C.6 Underfitting

- **Definition**: a model performs poorly on both training data and unseen data.
- **Observed here**: not present, because both train and validation metrics are high.

## C.7 Class Imbalance

- **Definition**: classes do not appear with roughly equal frequency.
- **Why it matters here**: many account labels have very low support, so accuracy alone is not enough.

## C.8 Generalization

- **Definition**: the ability of a model to perform well on new examples.
- **Why it matters here**: the finance use case depends on unseen future transactions, not memorized templates.

## C.9 Bias-Variance Trade-off

- **Bias**: error from an overly simple model.
- **Variance**: error from excessive sensitivity to training details.
- **Observed here**: variance is the main issue, not bias.


# Appendix D: Indexed Sample And Split Definitions

## D.1 Sample

- **Definition**: a single row or observation in the dataset.
- **In this notebook**: one expense record is one sample.

## D.2 Training Sample

- **Definition**: a sample used to estimate model parameters.

## D.3 Validation Sample

- **Definition**: a held-out sample used to estimate generalization during development.

## D.4 Balanced Unseen Sample

- **Definition**: a held-out sample constructed so that eligible labels contribute equally many examples.
- **Purpose**: stress-test minority-class performance without changing the real production distribution.

## D.5 Eligible Label

- **Definition**: a label with enough rows and enough distinct normalized item-name groups to support balanced unseen evaluation.

## D.6 Support

- **Definition**: the number of evaluated samples belonging to a class.

## D.7 Leakage

- **Definition**: information from training data reaches evaluation data in a way that inflates the metric.
- **In this project**: repeated normalized item names can leak phrase templates across random row splits.


# Appendix E: Source Notes And Index

## E.1 Documentation Basis

The appendix definitions above are paraphrased from official documentation and glossary pages for:

1. pandas documentation
2. NumPy documentation
3. Matplotlib documentation
4. scikit-learn documentation
5. joblib documentation
6. Google Machine Learning glossary / crash course

## E.2 Index

| Index | Topic |
| --- | --- |
| A | Mathematical notes and equations |
| B | Machine-learning and Python library definitions |
| C | Core ML concept definitions |
| D | Sample and split terminology |
| E | Source notes and index |

## E.3 Usage Justification Summary

- pandas: best fit for labeled tabular data loading and summarization
- NumPy: numerical base layer for arrays and vectorized operations
- Matplotlib: reproducible report-quality charts
- scikit-learn: efficient classical ML pipeline for sparse text classification
- joblib: lightweight persistence for trained artifacts
